# SDM invullen

### Imports

In [1]:
import sqlite3
import pandas as pd
import os

### Verbinding maken met SDM

In [2]:
db_path = sqlite3.connect("../../Database/SDM1.db")

# Cursor = object that allows interaction with the database by executing SQL commands and fetching query results
cursor = db_path.cursor()

### Bron databases openen

In [3]:
bron1 = sqlite3.connect("../../Database/BikeToDrive_1_Accessoireverkoop.db")
bron2 = sqlite3.connect("../../Database/BikeToDrive_2_Fietsverkoop.db")
bron3 = sqlite3.connect("../../Database/BikeToDrive_3_Onderhoud.db")
bron4 = sqlite3.connect("../../Database/BikeToDrive_4_Accessoire_Inkoop.db")
bron5 = sqlite3.connect("../../Database/BikeToDrive_5_Fiets_Inkoop.db")

### SDM schema dictionary

In [4]:
# voor nu even uitzetten, we gebruiken SDM_SCHEMA in een lijst hier onder 

# SDM_SCHEMA = {
#     "Accessoire" :{
#         "accessoirenr" : "INT",
#         "naam" : "VARCHAR",
#         "standaardprijs" : "float",
#         "inkoopprijs": "float",
#         "soort" : "VARCHAR",
#         "leverancier": "INT"
#     },
#     "Accessoire_Inkoop":{
#         "inkoopnr" : "INT",
#         "inkoopmaand": "INT",
#         "inkoopjaar": "INT",
#         "aantal": "INT",
#         "accessoire": "INT"
#     }, 
#     "Accessoireverkoop_Accessoire_Verkoop":{
#         "äccessoire_verkoopnr" : "INT",
#         "datun" : "VARCHAR",
#         "aantal": "INT",
#         "verkoopprijs" : "float",
#         "Klant" : "INT",
#         "accessoire" : "INT",
#         "monteur" : "ÏNT"
#     },
#     "Fabrikant" :{
#         "fabrikantnr" : "INT",
#         "naam" : "VARCHAR",
#         "adres" : "VARCHAR",
#         "plaats" : "VARCHAR"
#     },
#     "Fiets" :{
#         "fietsnr" : "INT",
#         "soort" : "VARCHAR",
#         "merk" : "VARCHAR",
#         "type" : "VARCHAR",
#         "standaartprijs" : "INT",
#         "inkoopprijs" : "INT",
#         "kleur" : "VARCHAR",
#         "fabrikant" : "VARCHAR"
#     },
#     "Fiets_Inkoop":{
#         "inkoopnr" : "INT",
#         "inkoopmaand" : "INT",
#         "inkoopjaar" : "INT",
#         "aantal": "INT",
#         "fiets" : "INT",
#     },
#     "Fietsverkoop_Fiets_Verkoop" : {
#         "Fiets_verkoopnr" : "INT",
#         "datum" : "INT",
#         "aantal": "INT",
#         "verkoopprijs" : "float",
#         "klant" : "INT",
#         "fiets" : "INT",
#         "monteur" : "INT"
#     },
#     "Filiaal" : {
#         "filialnr" : "INT",
#         "naam" : "VARCHAR",
#         "adres" : "VARCHAR",
#         "provincie" : "VARCHAR"
#     },
#     "Klant" : {
#         "klantnr" : "INT",
#         "naam" : "VARCHAR",
#         "adres" : "VARCHAR",
#         "woonplaats" : "VARCHAR",
#         "geslacht" : "VARCHAR",
#         "geboortedatum" : "VARCHAR" 
#     },
#     "Leverancier" : {
#         "leveranciernr" : "INT",
#         "naam" : "VARCHAR",
#         "adres" : "VARCHAR",
#         "woonplaats" : "VARCHAR"
#     },
#     "Monteur" : {
#         "monteurnr" : "INT",
#         "naam" : "VARCHAR",
#         "woonplaats" : "VARCHAR",
#         "uurloon" : "FLOAT",
#         "filiaal" : "INT"
#     },
#     "Onderhoud" : {
#         "onderhoudnr" : "INT",
#         "datum" : "DATE",
#         "starttijd" : "TIME",
#         "eindtijd" : "TIME",
#         "fiets" : "INT",
#         "monteur" : "INT"
#     }
# }

Dit zorgt er voor dat we later makelijker dingen kunnen doen zoals:
1. automatisch tabellen maken
2. automatisch tabellen leegmaken
3. automatisch data laden

### Reset_knop
a.	Alle tabellen uit het SDM leegmaken. Dit is dus een soort “reset-knop” voor de invulling van je SDM-tabellen.

In [5]:

# Dit pakt alle tabelnamen uit onze schema
SDM_SCHEMA = {
    "Accessoire": {},
    "Accessoire_Inkoop": {},
    "Accessoireverkoop_Accessoire_Verkoop": {},
    "Fabrikant": {},
    "Fiets": {},
    "Fiets_Inkoop": {},
    "Fietsverkoop_Fiets_Verkoop": {},
    "Filiaal": {},
    "Klant": {},
    "Leverancier": {},
    "Monteur": {},
    "Onderhoud": {}
}
alle_tabellen = list(SDM_SCHEMA.keys())

# Je draait hier de volgorde om vanwegen foreign keys
alle_tabellen = alle_tabellen[::-1]

for tabel in alle_tabellen:
     # Dit is de error handling
    delete_statement = f"DELETE FROM {tabel};"
    try:
        cursor.execute(delete_statement)
        print(f"{tabel} geleegd")
        # Dit print de error voor debuggen
    except sqlite3.Error as e:
        print(f"FAILED: {delete_statement}")
        print(e)
        continue

db_path.commit()

Onderhoud geleegd
Monteur geleegd
Leverancier geleegd
Klant geleegd
Filiaal geleegd
Fietsverkoop_Fiets_Verkoop geleegd
Fiets_Inkoop geleegd
Fiets geleegd
Fabrikant geleegd
Accessoireverkoop_Accessoire_Verkoop geleegd
Accessoire_Inkoop geleegd
Accessoire geleegd


### Data uit bron lezen

In [6]:
# leverancier aleen uit "BikeToDrive_1_Accessoireverkoop", omdat bijde tabelen kwa inhoud presies het zelfte zijn.
df_leverancier = pd.read_sql_query(
    "SELECT * FROM Leverancier",
    bron1
)

# accessoire aleen uit "BikeToDrive_4_Accessoire_Inkoop", omdat die al de volledige dataset bevat.
df_accessoire = pd.read_sql_query(
    "SELECT * FROM Accessoire",
    bron4
)

df_accessoire_inkoop = pd.read_sql_query(
    "SELECT * FROM Accessoire_Inkoop",
    bron4
)

# filiaal aleen uit "BikeToDrive_3_Onderhoud", omdat die al de volledige dataset bevat
df_filiaal = pd.read_sql_query(
    "SELECT * FROM Filiaal",
    bron3
)

#fabrikant alleen uit "BikeToDrive_3_Onderhoud", omdat die al de volledige dataset bevat
df_fabrikant = pd.read_sql_query(
    "SELECT * FROM Fabrikant",
    bron3
)

#klant alleen uit "BikeToDrive_2_Fiets verkoop", omdat die al de meeste dataset bevat
df_klant = pd.read_sql_query(
    "SELECT * FROM Klant",
    bron2
)

#monteur alleen uit "BikeToDrive_3_Onderhoud", omdat die al de meeste dataset bevat
df_monteur = pd.read_sql_query(
    "SELECT * FROM Monteur",
    bron3
)

#fiets uit "BikeToDrive_2_Fiets verkoop", kan ook uit fiets_inkoop, omdat bijde tabelen kwa inhoud presies het zelfte zijn.
df_fiets = pd.read_sql_query(
    "SELECT * FROM Fiets",
    bron2
)

#fiets_inkoop uit "BikeToDrive_5_Fiets_Inkoop", tabel bestaat alleen in deze database.
df_fiets_inkoop = pd.read_sql_query(
    "SELECT * FROM Fiets_Inkoop",
    bron5
)

#accessoire_verkoop uit "BikeToDrive_1_Accessoireverkoop", tabel bestaat alleen in deze database.
df_accessoire_verkoop = pd.read_sql_query(
    "SELECT * FROM Accessoire_Verkoop",
    bron1
)

#fiets_verkoop alleen uit "BikeToDrive_2_Fiets verkoop", omdat die al de meeste dataset bevat
df_fiets_verkoop = pd.read_sql_query(
    "SELECT * FROM Fiets_Verkoop",
    bron2
)

# onderhoud aleen uit "BikeToDrive_3_Onderhoud", tabel bestaat alleen in deze database, heeft de volledige dataset.
df_onderhoud = pd.read_sql_query(
    "SELECT * FROM Onderhoud",
    bron3
)


### Data overzetten naar SDM
b.	Data van elk .db-bestand overzetten naar het Source Data Model. Besteed hierbij extra aandacht aan de database-overschrijdende associaties.


In [ ]:

#data inladen naar SDM
#append => data toevoegen aan de bestaande tabel
#index=false => voorkomt dat de index als extra kolom in de database wordt opgeslagen

df_leverancier.to_sql("Leverancier", db_path, if_exists="append", index=False)

df_filiaal.to_sql("Filiaal", db_path, if_exists="append", index=False)

df_fabrikant.to_sql("Fabrikant", db_path, if_exists="append", index=False)

df_klant.to_sql("Klant", db_path, if_exists="append", index=False)

df_monteur.to_sql("Monteur", db_path, if_exists="append", index=False)

df_accessoire.to_sql("Accessoire", db_path, if_exists="append", index=False)

df_fiets.to_sql("Fiets", db_path, if_exists="append", index=False)

df_accessoire_inkoop.to_sql("Accessoire_Inkoop", db_path, if_exists="append", index=False)

df_fiets_inkoop.to_sql("Fiets_Inkoop", db_path, if_exists="append", index=False)

df_accessoire_verkoop.to_sql("Accessoireverkoop_Accessoire_Verkoop", db_path, if_exists="append", index=False)

df_fiets_verkoop.to_sql("Fietsverkoop_Fiets_Verkoop", db_path, if_exists="append", index=False)

df_onderhoud.to_sql("Onderhoud", db_path, if_exists="append", index=False)

db_path.commit()
print("data ingeladen")


data ingeladen


### 6. testing
Test je Jupyter Notebook door in de operationele databronnen bepaalde wijzigingen aan te brengen (rijen toevoegen, updaten of verwijderen) en vervolgens te kijken of het SDM deze updates volledig overneemt na het uitvoeren van het Jupyter Notebook.

insert test


In [ ]:
#cursor voor alleen bron3 => "Database/BikeToDrive_3_Onderhoud.db"
cursor_bron3 = bron3.cursor() 

#insert via cursor => rij toegevoegd
cursor_bron3.execute("""
INSERT INTO Filiaal (filiaalnr, naam, adres, provincie)
VALUES (123, 'Test Filiaal', 'appel straat 11', 'Den Haag')
""")
bron3.commit()
print("nieuwe filiaal togevoegd")


nieuwe filiaal togevoegd


update test

In [ ]:
#update via cursor => rij updated van filiaal
cursor_bron3.execute("""
UPDATE Filiaal
SET naam = 'New filiaal'
WHERE filiaalnr = 123
""")

bron3.commit()
print("Filiaal updated")

Filiaal updated


delete Test

In [ ]:
#delete via cursor => rij verwijderd van filiaal
cursor_bron3.execute("""
DELETE FROM Filiaal
WHERE filiaalnr = 123
""")
bron3.commit()
print("Filiaal verwijderd")

Filiaal verwijderd


kijken of het SDM updates volledig overneemt 

In [ ]:
#leest alle rijen bron3/"Database/BikeToDrive_3_Onderhoud.db" => filiaal
df_filiaal = pd.read_sql_query("SELECT * FROM Filiaal", bron3)

#resultaat
#replace omdat alles verwijder moet en de nieuwe data erin wordt gezet
df_filiaal.to_sql("Filiaal", db_path, if_exists="replace", index=False)

7

In [ ]:
#data inlezen van filiaal => alleen rij met juiste id
df_check = pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr=123", db_path)

#verwachte uitkomst is dat de rij verwijderd is
print(df_check)


Empty DataFrame
Columns: [filiaalnr, naam, adres, provincie]
Index: []
